# 13.3 · LightGBM Hyperparameter Tuning (Optuna)

**Purpose**: Bayesian search over LightGBM hyperparameters using
Optuna to find a configuration that minimises val MAE below the
fixed-param baseline in notebook 13.

| I/O | Path |
|-----|------|
| Input  | `data/processed/train.parquet` |
| Input  | `data/processed/val.parquet` |
| Input  | `data/processed/feature_manifest.json` |
| Output | `model/lgbm_tuned_model.pkl` |
| Output | `model/lgbm_tuned_metadata.json` |
| Output | `model/optuna_study.pkl` |

**Configure** `N_TRIALS` to control search budget.

---
## 1 · Setup

In [ ]:
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import optuna
from pathlib import Path
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.rcParams.update({'figure.dpi': 120})

SEED     = 42
N_TRIALS = 50   # <-- increase for a more thorough search

TRAIN_PATH    = Path('../data/processed/train.parquet')
VAL_PATH      = Path('../data/processed/val.parquet')
MANIFEST_PATH = Path('../data/processed/feature_manifest.json')
MODEL_DIR     = Path('../model')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_MAE = 9.842  # from notebook 13 fixed-param run

print(f'Optuna version : {optuna.__version__}')
print(f'N_TRIALS       : {N_TRIALS}')

---
## 2 · Load Data

In [ ]:
with open(MANIFEST_PATH, 'r', encoding='utf-8') as fh:
    manifest = json.load(fh)

NUMERIC_FEATURES = manifest['numeric_features']
CAT_FEATURES     = manifest['cat_features']
TARGET           = manifest['target']

df_train = pd.read_parquet(TRAIN_PATH)
df_val   = pd.read_parquet(VAL_PATH)

for col in CAT_FEATURES:
    for split in [df_train, df_val]:
        if col in split.columns:
            split[col] = split[col].astype('category')

FEATURE_COLS = [
    c for c in NUMERIC_FEATURES + CAT_FEATURES
    if c in df_train.columns
]

X_train = df_train[FEATURE_COLS]
X_val   = df_val[FEATURE_COLS].copy()
y_train = df_train[TARGET].values
y_val   = df_val[TARGET].values

# Align category levels
for col in CAT_FEATURES:
    if col in X_train.columns:
        X_val[col] = pd.Categorical(
            X_val[col],
            categories=X_train[col].cat.categories,
        )

lgb_train = lgb.Dataset(
    X_train, label=y_train,
    categorical_feature=CAT_FEATURES,
    free_raw_data=False,
)
lgb_val = lgb.Dataset(
    X_val, label=y_val,
    categorical_feature=CAT_FEATURES,
    reference=lgb_train,
    free_raw_data=False,
)

print(f'Train : {len(df_train):,} rows')
print(f'Val   : {len(df_val):,} rows')
print(f'Features : {len(FEATURE_COLS)}')

---
## 3 · Define Objective & Run Study

In [ ]:
def objective(trial: optuna.Trial) -> float:
    """Return val MAE for a given hyperparameter sample."""
    params = {
        'objective':        'regression_l1',
        'metric':           'mae',
        'verbose':          -1,
        'seed':             SEED,
        'learning_rate': trial.suggest_float(
            'learning_rate', 0.005, 0.1, log=True
        ),
        'num_leaves': trial.suggest_int(
            'num_leaves', 31, 511
        ),
        'min_data_in_leaf': trial.suggest_int(
            'min_data_in_leaf', 20, 200
        ),
        'feature_fraction': trial.suggest_float(
            'feature_fraction', 0.5, 1.0
        ),
        'bagging_fraction': trial.suggest_float(
            'bagging_fraction', 0.5, 1.0
        ),
        'bagging_freq': trial.suggest_int(
            'bagging_freq', 1, 10
        ),
        'reg_alpha': trial.suggest_float(
            'reg_alpha', 1e-3, 10.0, log=True
        ),
        'reg_lambda': trial.suggest_float(
            'reg_lambda', 1e-3, 10.0, log=True
        ),
        'min_gain_to_split': trial.suggest_float(
            'min_gain_to_split', 0.0, 1.0
        ),
    }

    callbacks = [
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=-1),
    ]
    model = lgb.train(
        params,
        lgb_train,
        num_boost_round=3000,
        valid_sets=[lgb_val],
        valid_names=['valid'],
        callbacks=callbacks,
    )

    pred = np.clip(
        model.predict(X_val, num_iteration=model.best_iteration),
        a_min=0, a_max=None,
    )
    mae = mean_absolute_error(y_val, pred)
    trial.set_user_attr('best_iteration', model.best_iteration)
    return mae


study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=10),
)

print(f'Running {N_TRIALS} trials...')
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\nBest val MAE  : {study.best_value:.4f} min')
print(f'Baseline MAE  : {BASELINE_MAE:.4f} min')
print(
    f'Improvement   : '
    f'{(BASELINE_MAE - study.best_value) / BASELINE_MAE * 100:.1f}%'
)
print(f'\nBest params:')
for k, v in study.best_params.items():
    print(f'  {k:<25} {v}')

---
## 4 · Study Visualisations

In [ ]:
trials_df = study.trials_dataframe()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# ── Optimisation history ──────────────────────────────────────────────────
axes[0].plot(
    trials_df['number'],
    trials_df['value'],
    alpha=0.4, color='steelblue', label='Trial MAE',
)
axes[0].plot(
    trials_df['number'],
    trials_df['value'].cummin(),
    color='#06C167', linewidth=2, label='Best so far',
)
axes[0].axhline(
    BASELINE_MAE, color='#FF4B4B',
    linestyle='--', linewidth=1.5,
    label=f'Baseline {BASELINE_MAE:.2f}',
)
axes[0].set_title('Optimisation History', fontweight='bold')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Val MAE (min)')
axes[0].legend(fontsize=9)

# ── Parameter importance ──────────────────────────────────────────────────
importance = optuna.importance.get_param_importances(study)
imp_df = pd.DataFrame(
    importance.items(), columns=['param', 'importance']
).sort_values('importance')

axes[1].barh(
    imp_df['param'], imp_df['importance'],
    color='#06C167', edgecolor='none',
)
axes[1].set_title('Hyperparameter Importance', fontweight='bold')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

print('Top 5 trials:')
print(
    trials_df[['number', 'value']]
    .sort_values('value')
    .head(5)
    .to_string(index=False)
)

---
## 5 · Retrain with Best Params (full early stopping budget)

In [ ]:
best_params = {
    **study.best_params,
    'objective': 'regression_l1',
    'metric':    ['mae', 'rmse'],
    'verbose':   -1,
    'seed':      SEED,
}

callbacks = [
    lgb.early_stopping(stopping_rounds=100, verbose=False),
    lgb.log_evaluation(period=200),
]

print('Retraining with best params...')
tuned_model = lgb.train(
    best_params,
    lgb_train,
    num_boost_round=5000,
    valid_sets=[lgb_train, lgb_val],
    valid_names=['train', 'valid'],
    callbacks=callbacks,
)

from sklearn.metrics import mean_squared_error, r2_score

pred_val = np.clip(
    tuned_model.predict(
        X_val, num_iteration=tuned_model.best_iteration
    ),
    a_min=0, a_max=None,
)

final_mae  = mean_absolute_error(y_val, pred_val)
final_rmse = float(np.sqrt(mean_squared_error(y_val, pred_val)))
final_r2   = r2_score(y_val, pred_val)

print(f'\nTuned model results:')
print(f'  Best iteration : {tuned_model.best_iteration}')
print(f'  Val MAE        : {final_mae:.4f} min')
print(f'  Val RMSE       : {final_rmse:.4f} min')
print(f'  Val R²         : {final_r2:.4f}')
print(
    f'  vs baseline    : '
    f'{(BASELINE_MAE - final_mae) / BASELINE_MAE * 100:+.1f}%'
)

---
## 6 · Save Artifacts

In [ ]:
TUNED_MODEL_PATH = MODEL_DIR / 'lgbm_tuned_model.pkl'
TUNED_META_PATH  = MODEL_DIR / 'lgbm_tuned_metadata.json'
STUDY_PATH       = MODEL_DIR / 'optuna_study.pkl'

joblib.dump(tuned_model, TUNED_MODEL_PATH)
print(f'Tuned model saved → {TUNED_MODEL_PATH}')

joblib.dump(study, STUDY_PATH)
print(f'Optuna study saved → {STUDY_PATH}')

metadata = {
    'model':             'lightgbm_tuned',
    'n_trials':          N_TRIALS,
    'best_params':       study.best_params,
    'feature_cols':      FEATURE_COLS,
    'numeric_features':  NUMERIC_FEATURES,
    'cat_features':      CAT_FEATURES,
    'target':            TARGET,
    'best_iteration':    int(tuned_model.best_iteration),
    'val_mae':           round(final_mae, 4),
    'val_rmse':          round(final_rmse, 4),
    'val_r2':            round(final_r2, 4),
    'baseline_mae':      BASELINE_MAE,
    'improvement_pct':   round(
        (BASELINE_MAE - final_mae) / BASELINE_MAE * 100, 2
    ),
    'sla_threshold_min': manifest['sla_threshold_min'],
    'split_date_val_start':  manifest.get('split_date_val_start'),
    'split_date_test_start': manifest.get('split_date_test_start'),
}
with open(TUNED_META_PATH, 'w', encoding='utf-8') as fh:
    json.dump(metadata, fh, indent=2)
print(f'Metadata saved → {TUNED_META_PATH}')

print(
    f'\nFinal: MAE={final_mae:.4f}  '
    f'RMSE={final_rmse:.4f}  '
    f'R²={final_r2:.4f}'
)